# exp_013：Mask-aware StockMixer-Lite + 递归预测

## tl;dr

这是完整、自包含的单模型实验文件：数据读取、StockMixer-Lite、20 轮 GPU/FP16 训练、EMA、检查点恢复、Valid/Test 推理和 `0.8/0.2` 递归状态都在本 Notebook 内。

- 默认 `RUN_MODE = "smoke"`，只做真实数据单步训练和两期推理验证。
- 正式训练时将配置单元改为 `RUN_MODE = "full"` 后从头运行。
- 中断恢复时再将 `RESUME = True`。
- 不会覆盖 `04_results/final_submission/prediction.npy`。


## 1. 环境与固定配置

必须选择已安装的 Anaconda kernel **`jingge_ts`**。配置固定为单种子 42、20 epochs、60 期窗口、每截面最多 768 只股票和 FP16。


In [1]:
from __future__ import annotations

import argparse
import copy
import csv
import hashlib
import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import rankdata


T = 3603
S = 5282
TRAIN_START = 486
VALID_START = 2918
VALID_STOP = 3161
TEST_START = 3161
TEST_STOP = 3603


@dataclass(frozen=True)
class Config:
    seed: int = 42
    window: int = 60
    sequence_features: int = 40
    rank_features: int = 20
    state_features: int = 4
    categories: int = 33
    category_embedding: int = 8
    hidden: int = 64
    token_hidden: int = 128
    channel_hidden: int = 128
    mixer_blocks: int = 2
    dropout: float = 0.1
    epochs: int = 20
    time_bins: int = 64
    times_per_bin: int = 8
    stocks_per_time: int = 768
    inference_batch: int = 512
    pair_count: int = 4096
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    warmup_epochs: int = 2
    gradient_clip: float = 1.0
    ema_decay: float = 0.999
    recursive_alpha: float = 0.8
    ic_weight: float = 0.60
    huber_weight: float = 0.25
    pairwise_weight: float = 0.15
    pairwise_scale: float = 10.0

    @property
    def steps_per_epoch(self) -> int:
        return self.time_bins * self.times_per_bin

    @property
    def total_steps(self) -> int:
        return self.epochs * self.steps_per_epoch

    @property
    def warmup_steps(self) -> int:
        return self.warmup_epochs * self.steps_per_epoch


CONFIG = Config()


import sys

def find_project_root() -> Path:
    """Locate the project whether Jupyter starts in root or experiment dir."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data.z").exists() and (candidate / "03_cache" / "processed_data_v1").exists():
            return candidate
    raise RuntimeError("无法定位项目根目录：需要同时包含 data.z 和 03_cache/processed_data_v1")


PROJECT_ROOT = find_project_root()
DATASET_DIR = PROJECT_ROOT / "03_cache" / "processed_data_v1"
RESULT_DIR = PROJECT_ROOT / "04_results" / "exp_013_stockmixer_recursive"
EXPECTED_PYTHON = Path(r"D:\anaconda\anaconda_data\envs\jingge_ts\python.exe")
RUN_MODE = "full"  # 完整训练时仅改为 "full"
RESUME = False       # 完整训练中断后改为 True

if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError(
        f"当前解释器为 {sys.executable}，请切换 Notebook kernel 到 jingge_ts ({EXPECTED_PYTHON})"
    )
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE 只能是 smoke 或 full")
if RUN_MODE == "smoke" and RESUME:
    raise ValueError("RESUME 只适用于 RUN_MODE='full'")

print({
    "project_root": str(PROJECT_ROOT),
    "python": sys.executable,
    "torch": torch.__version__,
    "cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "run_mode": RUN_MODE,
    "resume": RESUME,
})


{'project_root': 'D:\\google_dl\\book\\友安杯', 'python': 'd:\\anaconda\\anaconda_data\\envs\\jingge_ts\\python.exe', 'torch': '2.6.0+cu124', 'cuda_build': '12.4', 'cuda_available': True, 'gpu': 'NVIDIA GeForce RTX 4060 Laptop GPU', 'run_mode': 'full', 'resume': False}


## 2. 缓存契约与数据读取

只读取 `processed_data_v1`：40 维序列、历史 mask，以及 tree 缓存中的当前排名、覆盖状态和 `cat_6`。窗口严格截止当期，不读取未来数据。


In [2]:
def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def file_sha256(path: Path, block_size: int = 16 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(block_size):
            digest.update(block)
    return digest.hexdigest()


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + ".partial")
    partial.write_text(text, encoding="utf-8")
    os.replace(partial, path)


def atomic_write_json(path: Path, payload) -> None:
    atomic_write_text(path, json.dumps(json_ready(payload), ensure_ascii=False, indent=2))


def atomic_save_npy(path: Path, array: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + ".partial")
    with partial.open("wb") as handle:
        np.save(handle, array)
    os.replace(partial, path)


def atomic_torch_save(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + ".partial")
    torch.save(payload, partial)
    os.replace(partial, path)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class Dataset:
    """Read-only access to the processed cache with explicit time group offsets."""

    split_bounds = {
        "train": (TRAIN_START, VALID_START),
        "valid": (VALID_START, VALID_STOP),
        "test": (TEST_START, TEST_STOP),
    }

    def __init__(self, dataset_dir: Path = DATASET_DIR):
        self.dir = Path(dataset_dir)
        ready_path = self.dir / "READY"
        manifest_path = self.dir / "manifest.json"
        if not ready_path.exists() or not manifest_path.exists():
            raise FileNotFoundError("processed_data_v1 缺少 READY 或 manifest.json")
        self.ready = json.loads(ready_path.read_text(encoding="utf-8"))
        self.manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if self.ready["manifest_sha256"] != file_sha256(manifest_path):
            raise RuntimeError("manifest.json 的 SHA-256 与 READY 不一致")
        self._validate_manifest()
        self.sequence = np.load(self.dir / "sequence" / "X.npy", mmap_mode="r")
        self.sequence_mask = np.load(self.dir / "sequence" / "mask_x.npy", mmap_mode="r")
        self.common: dict[str, dict[str, np.ndarray]] = {}
        self.tree: dict[str, np.ndarray] = {}
        self.offsets: dict[str, np.ndarray] = {}
        for split in ("train", "valid", "test"):
            common_dir = self.dir / "common"
            entry = {
                "time": np.load(common_dir / f"{split}_time.npy", mmap_mode="r"),
                "stock": np.load(common_dir / f"{split}_stock.npy", mmap_mode="r"),
                "groups": np.load(common_dir / f"{split}_group_sizes.npy", mmap_mode="r"),
            }
            if split != "test":
                entry["y"] = np.load(common_dir / f"{split}_y.npy", mmap_mode="r")
            self.common[split] = entry
            self.tree[split] = np.load(self.dir / "tree" / f"{split}_X.npy", mmap_mode="r")
            groups = np.asarray(entry["groups"], dtype=np.int64)
            self.offsets[split] = np.concatenate(([0], np.cumsum(groups, dtype=np.int64)))
        self._validate_arrays()

    def _validate_manifest(self) -> None:
        manifest = self.manifest
        expected_dimensions = {"time": T, "stock": S, "raw_numeric": 99, "raw_category": 9}
        if manifest.get("status") != "ready" or manifest.get("dimensions") != expected_dimensions:
            raise RuntimeError("processed_data_v1 manifest 契约不匹配")
        if manifest["features"]["sequence_count"] != CONFIG.sequence_features:
            raise RuntimeError("sequence 特征数不是 40")
        if manifest["features"]["tree_count"] <= 414:
            raise RuntimeError("tree 视图缺少 cat_6 列")
        if manifest["category_state"]["unknown_codes"][6] != 32:
            raise RuntimeError("cat_6 unknown code 不是预期的 32")
        for split, (start, stop) in self.split_bounds.items():
            meta = manifest["splits"][split]
            if int(meta["start"]) != start or int(meta["stop"]) != stop:
                raise RuntimeError(f"{split} 时间边界不匹配")

    def _validate_arrays(self) -> None:
        if self.sequence.shape != (T, S, CONFIG.sequence_features):
            raise RuntimeError(f"sequence/X.npy shape 异常: {self.sequence.shape}")
        if self.sequence.dtype != np.float32 or self.sequence_mask.shape != (T, S):
            raise RuntimeError("sequence 缓存 dtype 或 mask shape 异常")
        for split, (start, stop) in self.split_bounds.items():
            expected_rows = int(self.manifest["expected_rows"][split])
            entry = self.common[split]
            if self.tree[split].shape != (expected_rows, 419):
                raise RuntimeError(f"{split} tree shape 异常: {self.tree[split].shape}")
            if int(self.offsets[split][-1]) != expected_rows:
                raise RuntimeError(f"{split} group sum 异常")
            if len(entry["groups"]) != stop - start:
                raise RuntimeError(f"{split} group 时间长度异常")
            if len(entry["time"]) != expected_rows or len(entry["stock"]) != expected_rows:
                raise RuntimeError(f"{split} common 行数异常")

    def row_bounds(self, split: str, time_index: int) -> tuple[int, int]:
        start, stop = self.split_bounds[split]
        if not start <= time_index < stop:
            raise IndexError(f"{split} 不包含时间 {time_index}")
        local_time = time_index - start
        return int(self.offsets[split][local_time]), int(self.offsets[split][local_time + 1])

    def sequence_window(self, time_index: int, stocks: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """Return [N, 60, 40] data ending exactly at ``time_index``."""
        stocks = np.asarray(stocks, dtype=np.int64)
        window_start = max(0, time_index - CONFIG.window + 1)
        observed_length = time_index - window_start + 1
        left_pad = CONFIG.window - observed_length
        values = np.zeros((stocks.size, CONFIG.window, CONFIG.sequence_features), dtype=np.float32)
        mask = np.zeros((stocks.size, CONFIG.window), dtype=np.float32)
        block = np.asarray(self.sequence[window_start:time_index + 1, stocks, :], dtype=np.float32)
        block_mask = np.asarray(self.sequence_mask[window_start:time_index + 1, stocks], dtype=np.float32)
        values[:, left_pad:] = np.transpose(block, (1, 0, 2))
        mask[:, left_pad:] = block_mask.T
        return values, mask

    def time_batch(
        self,
        split: str,
        time_index: int,
        cap: int | None = None,
        rng: np.random.Generator | None = None,
        require_target: bool = True,
    ) -> dict[str, np.ndarray]:
        row_start, row_stop = self.row_bounds(split, time_index)
        full_size = row_stop - row_start
        if full_size <= 0:
            raise RuntimeError(f"{split} t={time_index} 没有可用股票")
        local_positions = np.arange(full_size, dtype=np.int64)
        target_rank = None
        if require_target:
            if "y" not in self.common[split]:
                raise RuntimeError(f"{split} 没有标签")
            full_target = np.asarray(self.common[split]["y"][row_start:row_stop], dtype=np.float32)
            if not np.isfinite(full_target).all():
                raise RuntimeError(f"{split} t={time_index} 标签包含非有限值")
            target_rank = (rankdata(full_target, method="average") / float(full_size)).astype(np.float32)
        if cap is not None and full_size > cap:
            if rng is None:
                local_positions = np.linspace(0, full_size - 1, int(cap), dtype=np.int64)
            else:
                local_positions = np.sort(rng.choice(full_size, size=int(cap), replace=False)).astype(np.int64)
        rows = row_start + local_positions
        stocks = np.asarray(self.common[split]["stock"][rows], dtype=np.int64)
        sequence, history_mask = self.sequence_window(time_index, stocks)
        tree = np.asarray(self.tree[split][rows], dtype=np.float32)
        categories = np.rint(tree[:, 414]).astype(np.int64)
        categories = np.clip(categories, 0, CONFIG.categories - 1)
        batch = {
            "sequence": sequence,
            "history_mask": history_mask,
            "current_rank": np.ascontiguousarray(tree[:, 40:60], dtype=np.float32),
            "state": np.ascontiguousarray(tree[:, 324:328], dtype=np.float32),
            "category": categories,
            "stock": stocks,
            "row": rows,
        }
        if target_rank is not None:
            batch["target"] = target_rank[local_positions]
        return batch

    def evaluation_mask(self, split: str) -> np.ndarray:
        start, stop = self.split_bounds[split]
        mask = np.zeros((stop - start, S), dtype=bool)
        times = np.asarray(self.common[split]["time"], dtype=np.int64)
        stocks = np.asarray(self.common[split]["stock"], dtype=np.int64)
        mask[times - start, stocks] = True
        return mask


## 3. Mask-aware StockMixer-Lite

In [3]:
class MixerBlock(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.token_norm = nn.LayerNorm(config.hidden)
        self.token_mlp = nn.Sequential(
            nn.Linear(config.window, config.token_hidden),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.token_hidden, config.window),
            nn.Dropout(config.dropout),
        )
        self.channel_norm = nn.LayerNorm(config.hidden)
        self.channel_mlp = nn.Sequential(
            nn.Linear(config.hidden, config.channel_hidden),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.channel_hidden, config.hidden),
            nn.Dropout(config.dropout),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        token_input = self.token_norm(inputs).transpose(1, 2)
        inputs = inputs + self.token_mlp(token_input).transpose(1, 2)
        return inputs + self.channel_mlp(self.channel_norm(inputs))


class StockMixerLite(nn.Module):
    def __init__(self, config: Config = CONFIG):
        super().__init__()
        self.config = config
        self.input_projection = nn.Linear(config.sequence_features + 1, config.hidden)
        self.blocks = nn.ModuleList([MixerBlock(config) for _ in range(config.mixer_blocks)])
        self.temporal_projection = nn.Sequential(
            nn.LayerNorm(config.hidden * 4),
            nn.Linear(config.hidden * 4, config.hidden),
            nn.GELU(),
            nn.Dropout(config.dropout),
        )
        self.category_embedding = nn.Embedding(config.categories, config.category_embedding)
        head_input = config.hidden * 6 + config.rank_features + config.state_features + config.category_embedding
        self.head = nn.Sequential(
            nn.LayerNorm(head_input),
            nn.Linear(head_input, 128),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(64, 1),
        )

    @staticmethod
    def _masked_mean(hidden: torch.Tensor, mask: torch.Tensor, length: int) -> torch.Tensor:
        hidden = hidden[:, -length:, :]
        weights = mask[:, -length:].unsqueeze(-1)
        denominator = weights.sum(dim=1).clamp_min(1.0)
        return (hidden * weights).sum(dim=1) / denominator

    def encode(self, sequence: torch.Tensor, history_mask: torch.Tensor) -> torch.Tensor:
        inputs = torch.cat((sequence, history_mask.unsqueeze(-1)), dim=-1)
        hidden = self.input_projection(inputs)
        for block in self.blocks:
            hidden = block(hidden)
        pooled = torch.cat(
            (
                hidden[:, -1, :],
                self._masked_mean(hidden, history_mask, 5),
                self._masked_mean(hidden, history_mask, 20),
                self._masked_mean(hidden, history_mask, 60),
            ),
            dim=-1,
        )
        return self.temporal_projection(pooled)

    def predict_from_embeddings(
        self,
        stock_state: torch.Tensor,
        current_rank: torch.Tensor,
        state_features: torch.Tensor,
        category: torch.Tensor,
    ) -> torch.Tensor:
        # Cross-sectional reductions span up to thousands of stocks.  Keep
        # these accumulations in FP32 even when the encoder runs under AMP.
        stock_state = stock_state.float()
        current_rank = current_rank.float()
        state_features = state_features.float()
        market_mean = stock_state.mean(dim=0, keepdim=True)
        market_std = stock_state.std(dim=0, keepdim=True, unbiased=False)
        expanded_mean = market_mean.expand_as(stock_state)
        expanded_std = market_std.expand_as(stock_state)
        category_sums = stock_state.new_zeros((self.config.categories, self.config.hidden))
        category_counts = stock_state.new_zeros((self.config.categories, 1))
        category_sums.index_add_(0, category, stock_state)
        category_counts.index_add_(0, category, stock_state.new_ones((stock_state.shape[0], 1)))
        category_means = category_sums / category_counts.clamp_min(1.0)
        group_state = category_means[category]
        features = torch.cat(
            (
                stock_state,
                expanded_mean,
                expanded_std,
                group_state,
                stock_state - expanded_mean,
                stock_state - group_state,
                current_rank,
                state_features,
                self.category_embedding(category),
            ),
            dim=-1,
        )
        return torch.sigmoid(self.head(features).squeeze(-1))

    def forward(
        self,
        sequence: torch.Tensor,
        history_mask: torch.Tensor,
        current_rank: torch.Tensor,
        state_features: torch.Tensor,
        category: torch.Tensor,
    ) -> torch.Tensor:
        stock_state = self.encode(sequence, history_mask)
        return self.predict_from_embeddings(stock_state, current_rank, state_features, category)


class ModelEMA:
    def __init__(self, model: nn.Module, decay: float):
        self.decay = float(decay)
        self.model = copy.deepcopy(model).eval()
        for parameter in self.model.parameters():
            parameter.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        source = model.state_dict()
        for name, value in self.model.state_dict().items():
            incoming = source[name].detach()
            if value.is_floating_point():
                value.mul_(self.decay).add_(incoming, alpha=1.0 - self.decay)
            else:
                value.copy_(incoming)

    def state_dict(self):
        return self.model.state_dict()

    def load_state_dict(self, state_dict) -> None:
        self.model.load_state_dict(state_dict)


## 4. 损失、训练与断点恢复

损失固定为 `0.60×相关性 + 0.25×Huber + 0.15×Pairwise`。EMA decay 固定为 0.999；GradScaler 初始值 1024 已在 RTX 4060 的真实批次上验证。


In [4]:
def correlation_loss(prediction: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    prediction_centered = prediction - prediction.mean()
    target_centered = target - target.mean()
    numerator = (prediction_centered * target_centered).mean()
    denominator = prediction_centered.square().mean().sqrt() * target_centered.square().mean().sqrt()
    return 1.0 - numerator / denominator.clamp_min(1e-6)


def composite_loss(
    prediction: torch.Tensor,
    target: torch.Tensor,
    config: Config = CONFIG,
) -> tuple[torch.Tensor, dict[str, float]]:
    ic = correlation_loss(prediction.float(), target.float())
    huber = F.huber_loss(prediction.float(), target.float(), delta=0.1)
    sample_count = prediction.numel()
    pair_i = torch.randint(0, sample_count, (config.pair_count,), device=prediction.device)
    pair_j = torch.randint(0, sample_count, (config.pair_count,), device=prediction.device)
    direction = torch.sign(target[pair_i] - target[pair_j])
    usable = direction != 0
    if bool(usable.any()):
        margin = (prediction[pair_i[usable]] - prediction[pair_j[usable]]) * direction[usable]
        pairwise = F.softplus(-config.pairwise_scale * margin.float()).mean()
    else:
        pairwise = prediction.float().sum() * 0.0
    total = config.ic_weight * ic + config.huber_weight * huber + config.pairwise_weight * pairwise
    metrics = {
        "loss": float(total.detach().cpu()),
        "ic_loss": float(ic.detach().cpu()),
        "huber_loss": float(huber.detach().cpu()),
        "pairwise_loss": float(pairwise.detach().cpu()),
    }
    return total, metrics


def to_device(batch: dict[str, np.ndarray], device: torch.device, include_target: bool = True):
    output = {
        "sequence": torch.from_numpy(batch["sequence"]).to(device, non_blocking=True),
        "history_mask": torch.from_numpy(batch["history_mask"]).to(device, non_blocking=True),
        "current_rank": torch.from_numpy(batch["current_rank"]).to(device, non_blocking=True),
        "state": torch.from_numpy(batch["state"]).to(device, non_blocking=True),
        "category": torch.from_numpy(batch["category"]).to(device, non_blocking=True),
    }
    if include_target:
        output["target"] = torch.from_numpy(batch["target"]).to(device, non_blocking=True)
    return output


def forward_batch(model: StockMixerLite, tensors: dict[str, torch.Tensor]) -> torch.Tensor:
    return model(
        tensors["sequence"],
        tensors["history_mask"],
        tensors["current_rank"],
        tensors["state"],
        tensors["category"],
    )


def sample_training_times(epoch: int, config: Config = CONFIG) -> np.ndarray:
    rng = np.random.default_rng(config.seed + 10_007 * epoch)
    bins = np.array_split(np.arange(TRAIN_START, VALID_START, dtype=np.int64), config.time_bins)
    selected: list[int] = []
    for time_bin in bins:
        count = min(config.times_per_bin, time_bin.size)
        selected.extend(rng.choice(time_bin, size=count, replace=False).tolist())
    values = np.asarray(selected, dtype=np.int64)
    rng.shuffle(values)
    if values.size != config.steps_per_epoch:
        raise RuntimeError("每轮时间抽样数不符合配置")
    return values


def lr_multiplier(step: int, config: Config = CONFIG) -> float:
    if step < config.warmup_steps:
        return float(step + 1) / float(config.warmup_steps)
    progress = (step - config.warmup_steps) / max(1, config.total_steps - config.warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))


def checkpoint_payload(
    epoch: int,
    global_step: int,
    model: StockMixerLite,
    ema: ModelEMA,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler,
    history: list[dict],
) -> dict:
    payload = {
        "epoch": int(epoch),
        "global_step": int(global_step),
        "config": asdict(CONFIG),
        "model": model.state_dict(),
        "ema": ema.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "history": history,
        "torch_rng_state": torch.get_rng_state(),
        "numpy_random_state": np.random.get_state(),
        "python_random_state": random.getstate(),
    }
    if torch.cuda.is_available():
        payload["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()
    return payload


def restore_checkpoint(path: Path, model, ema, optimizer, scheduler, scaler, device) -> tuple[int, int, list[dict]]:
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    if checkpoint.get("config") != asdict(CONFIG):
        raise RuntimeError("检查点配置与当前固定配置不一致")
    model.load_state_dict(checkpoint["model"])
    ema.load_state_dict(checkpoint["ema"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    scaler.load_state_dict(checkpoint["scaler"])
    torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
    np.random.set_state(checkpoint["numpy_random_state"])
    random.setstate(checkpoint["python_random_state"])
    if torch.cuda.is_available() and "cuda_rng_state_all" in checkpoint:
        torch.cuda.set_rng_state_all([state.cpu() for state in checkpoint["cuda_rng_state_all"]])
    return int(checkpoint["epoch"]), int(checkpoint["global_step"]), list(checkpoint["history"])


def write_history(path: Path, history: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + ".partial")
    fieldnames = list(history[0]) if history else ["epoch"]
    with partial.open("w", newline="", encoding="utf-8-sig") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(history)
    os.replace(partial, path)


def train_full(
    dataset: Dataset,
    model: StockMixerLite,
    ema: ModelEMA,
    device: torch.device,
    result_dir: Path,
    resume: bool,
) -> list[dict]:
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda step: lr_multiplier(step))
    # The correlation term has a noticeably larger initial gradient than MSE.
    # 65536 (PyTorch's default) overflows on the first FP16 backward pass on the
    # target RTX 4060; 1024 was verified finite on the real training batch.
    scaler = torch.amp.GradScaler(
        "cuda", enabled=device.type == "cuda", init_scale=1024.0, growth_interval=2000
    )
    checkpoint_path = result_dir / "checkpoint_latest.pt"
    start_epoch, global_step, history = 0, 0, []
    if resume:
        if not checkpoint_path.exists():
            raise FileNotFoundError(f"--resume 指定但检查点不存在: {checkpoint_path}")
        start_epoch, global_step, history = restore_checkpoint(
            checkpoint_path, model, ema, optimizer, scheduler, scaler, device
        )
        print(f"从 epoch={start_epoch}, global_step={global_step} 恢复", flush=True)
    elif checkpoint_path.exists():
        raise FileExistsError("已存在完整训练检查点；请使用 --resume，或人工归档旧结果后重跑")

    amp_enabled = device.type == "cuda"
    for epoch_index in range(start_epoch, CONFIG.epochs):
        epoch_started = time.time()
        model.train()
        totals = {"loss": 0.0, "ic_loss": 0.0, "huber_loss": 0.0, "pairwise_loss": 0.0}
        sample_total = 0
        times = sample_training_times(epoch_index)
        for step_in_epoch, time_index in enumerate(times, start=1):
            batch_rng = np.random.default_rng(CONFIG.seed + epoch_index * 100_003 + int(time_index))
            batch = dataset.time_batch(
                "train", int(time_index), cap=CONFIG.stocks_per_time, rng=batch_rng, require_target=True
            )
            tensors = to_device(batch, device, include_target=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                prediction = forward_batch(model, tensors)
                loss, components = composite_loss(prediction, tensors["target"])
            if not bool(torch.isfinite(loss)):
                raise FloatingPointError(f"epoch={epoch_index + 1} t={time_index} loss 非有限")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG.gradient_clip)
            if not bool(torch.isfinite(gradient_norm)):
                raise FloatingPointError(f"epoch={epoch_index + 1} t={time_index} gradient 非有限")
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            ema.update(model)
            global_step += 1
            count = int(batch["stock"].size)
            sample_total += count
            for key in totals:
                totals[key] += components[key]
            if step_in_epoch % 64 == 0:
                print(
                    f"epoch {epoch_index + 1:02d}/{CONFIG.epochs} "
                    f"step {step_in_epoch:03d}/{CONFIG.steps_per_epoch} "
                    f"loss={components['loss']:.5f} lr={optimizer.param_groups[0]['lr']:.2e}",
                    flush=True,
                )
        row = {
            "epoch": epoch_index + 1,
            "steps": CONFIG.steps_per_epoch,
            "samples": sample_total,
            "loss": totals["loss"] / CONFIG.steps_per_epoch,
            "ic_loss": totals["ic_loss"] / CONFIG.steps_per_epoch,
            "huber_loss": totals["huber_loss"] / CONFIG.steps_per_epoch,
            "pairwise_loss": totals["pairwise_loss"] / CONFIG.steps_per_epoch,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "seconds": time.time() - epoch_started,
        }
        history.append(row)
        write_history(result_dir / "training_history.csv", history)
        atomic_torch_save(
            checkpoint_path,
            checkpoint_payload(epoch_index + 1, global_step, model, ema, optimizer, scheduler, scaler, history),
        )
        print(json.dumps(json_ready(row), ensure_ascii=False), flush=True)
    atomic_torch_save(
        result_dir / "model_ema.pt",
        {"config": asdict(CONFIG), "model": ema.state_dict(), "epochs": CONFIG.epochs},
    )
    return history


## 5. 推理、递归状态与输出契约

每个截面先得到百分位秩 `q_t`。仅当股票连续两期有效时使用 `s_t=0.8q_t+0.2s_{t-1}`；首次出现或中断重现直接使用当前值。Valid/Test 各自独立初始化。


In [5]:
def predict_batch(
    model: StockMixerLite,
    batch: dict[str, np.ndarray],
    device: torch.device,
    amp_enabled: bool,
) -> np.ndarray:
    embeddings = []
    for start in range(0, batch["stock"].size, CONFIG.inference_batch):
        stop = min(start + CONFIG.inference_batch, batch["stock"].size)
        sequence = torch.from_numpy(batch["sequence"][start:stop]).to(device, non_blocking=True)
        history_mask = torch.from_numpy(batch["history_mask"][start:stop]).to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            embeddings.append(model.encode(sequence, history_mask))
    stock_state = torch.cat(embeddings, dim=0)
    current_rank = torch.from_numpy(batch["current_rank"]).to(device, non_blocking=True)
    state_features = torch.from_numpy(batch["state"]).to(device, non_blocking=True)
    category = torch.from_numpy(batch["category"]).to(device, non_blocking=True)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
        prediction = model.predict_from_embeddings(stock_state, current_rank, state_features, category)
    return prediction.float().cpu().numpy().astype(np.float32)


def percentile_rank(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        return np.empty(0, dtype=np.float32)
    return (rankdata(values, method="average") / float(values.size)).astype(np.float32)


def recursive_step(
    current_score: np.ndarray,
    stocks: np.ndarray,
    previous_state: np.ndarray,
    previous_active: np.ndarray,
    alpha: float = CONFIG.recursive_alpha,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    stocks = np.asarray(stocks, dtype=np.int64)
    q_value = percentile_rank(current_score)
    state_value = q_value.copy()
    continuing = previous_active[stocks]
    state_value[continuing] = (
        alpha * q_value[continuing] + (1.0 - alpha) * previous_state[stocks[continuing]]
    )
    final_rank = percentile_rank(state_value)
    next_state = np.zeros_like(previous_state, dtype=np.float32)
    next_active = np.zeros_like(previous_active, dtype=bool)
    next_state[stocks] = state_value
    next_active[stocks] = True
    return final_rank, next_state, next_active


def predict_split(
    dataset: Dataset,
    model: StockMixerLite,
    split: str,
    device: torch.device,
    time_limit: int | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    split_start, split_stop = dataset.split_bounds[split]
    if time_limit is not None:
        split_stop = min(split_stop, split_start + int(time_limit))
    full_time_count = dataset.split_bounds[split][1] - split_start
    raw_grid = np.full((full_time_count, S), 0.5, dtype=np.float32)
    recursive_grid = np.full((full_time_count, S), 0.5, dtype=np.float32)
    previous_state = np.zeros(S, dtype=np.float32)
    previous_active = np.zeros(S, dtype=bool)
    model.eval()
    for time_index in range(split_start, split_stop):
        batch = dataset.time_batch(split, time_index, cap=None, require_target=False)
        score = predict_batch(model, batch, device, amp_enabled=device.type == "cuda")
        if not np.isfinite(score).all():
            raise FloatingPointError(f"{split} t={time_index} 预测含非有限值")
        local_time = time_index - split_start
        raw_grid[local_time, batch["stock"]] = score
        final_rank, previous_state, previous_active = recursive_step(
            score, batch["stock"], previous_state, previous_active
        )
        recursive_grid[local_time, batch["stock"]] = final_rank
        if (local_time + 1) % 25 == 0 or time_index == split_stop - 1:
            print(f"{split} inference {local_time + 1}/{split_stop - split_start}", flush=True)
    return raw_grid, recursive_grid


def rank_ic(prediction: np.ndarray, target: np.ndarray) -> float:
    usable = np.isfinite(prediction) & np.isfinite(target)
    if int(usable.sum()) < 2:
        return float("nan")
    x_rank = rankdata(prediction[usable], method="average")
    y_rank = rankdata(target[usable], method="average")
    if x_rank.std() == 0 or y_rank.std() == 0:
        return float("nan")
    return float(np.corrcoef(x_rank, y_rank)[0, 1])


def score_grid(dataset: Dataset, split: str, grid: np.ndarray) -> dict:
    split_start, split_stop = dataset.split_bounds[split]
    if "y" not in dataset.common[split]:
        raise RuntimeError(f"{split} 没有标签")
    values = []
    for time_index in range(split_start, split_stop):
        row_start, row_stop = dataset.row_bounds(split, time_index)
        stocks = np.asarray(dataset.common[split]["stock"][row_start:row_stop], dtype=np.int64)
        target = np.asarray(dataset.common[split]["y"][row_start:row_stop], dtype=np.float32)
        values.append(rank_ic(grid[time_index - split_start, stocks], target))
    series = np.asarray(values, dtype=np.float64)
    finite = series[np.isfinite(series)]
    quarters = np.array_split(finite, 4)
    half = finite.size // 2
    return {
        "mean_rankic": float(np.nanmean(series)),
        "median_rankic": float(np.nanmedian(series)),
        "rankic_std": float(np.nanstd(series)),
        "positive_ratio": float(np.nanmean(series > 0)),
        "first_half_rankic": float(np.nanmean(series[:half])),
        "second_half_rankic": float(np.nanmean(series[half:])),
        "worst_quarter_rankic": float(min(np.nanmean(q) for q in quarters)),
        "time_count": int(finite.size),
    }


def validate_prediction_grid(grid: np.ndarray, evaluation_mask: np.ndarray) -> dict:
    if grid.shape != evaluation_mask.shape:
        raise AssertionError(f"预测 shape {grid.shape} != mask shape {evaluation_mask.shape}")
    if grid.dtype != np.float32:
        raise AssertionError(f"预测 dtype 必须为 float32，实际为 {grid.dtype}")
    if not np.isfinite(grid).all():
        raise AssertionError("预测包含 NaN/Inf")
    minimum, maximum = float(grid.min()), float(grid.max())
    if minimum < 0.0 or maximum > 1.0:
        raise AssertionError(f"预测范围异常: [{minimum}, {maximum}]")
    if not np.all(grid[~evaluation_mask] == np.float32(0.5)):
        raise AssertionError("非评估位置不是严格的 float32(0.5)")
    return {
        "shape": list(grid.shape),
        "dtype": str(grid.dtype),
        "minimum": minimum,
        "maximum": maximum,
        "evaluation_count": int(evaluation_mask.sum()),
        "non_evaluation_count": int((~evaluation_mask).sum()),
    }


def run_recursive_self_tests() -> dict:
    previous_state = np.zeros(4, dtype=np.float32)
    previous_active = np.zeros(4, dtype=bool)
    first, previous_state, previous_active = recursive_step(
        np.array([0.2, 0.8], dtype=np.float32), np.array([0, 1]), previous_state, previous_active
    )
    if not np.allclose(first, np.array([0.5, 1.0], dtype=np.float32)):
        raise AssertionError("首次出现递归规则错误")
    old_state = previous_state.copy()
    second, previous_state, previous_active = recursive_step(
        np.array([0.9], dtype=np.float32), np.array([0]), previous_state, previous_active
    )
    expected_state = CONFIG.recursive_alpha * 1.0 + (1.0 - CONFIG.recursive_alpha) * old_state[0]
    if not np.isclose(previous_state[0], expected_state) or not np.isclose(second[0], 1.0):
        raise AssertionError("连续股票递归规则错误")
    third, previous_state, previous_active = recursive_step(
        np.array([0.1, 0.9], dtype=np.float32), np.array([1, 2]), previous_state, previous_active
    )
    if not np.isclose(previous_state[1], 0.5) or not np.allclose(third, np.array([0.5, 1.0])):
        raise AssertionError("中断重现股票继承了陈旧状态")
    return {"first": first, "second": second, "third": third}


## 6. 冒烟测试与完整运行

In [6]:
def smoke_test(dataset: Dataset, device: torch.device, result_dir: Path) -> dict:
    smoke_dir = result_dir / "smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    if device.type != "cuda":
        raise RuntimeError("exp_013 冒烟测试要求 CUDA；当前未检测到可用 GPU")
    seed_everything(CONFIG.seed)
    model = StockMixerLite().to(device)
    ema = ModelEMA(model, CONFIG.ema_decay)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda step: lr_multiplier(step))
    scaler = torch.amp.GradScaler("cuda", enabled=True, init_scale=1024.0, growth_interval=2000)

    pad_values, pad_mask = dataset.sequence_window(0, np.array([0, 1], dtype=np.int64))
    if pad_values.shape != (2, 60, 40) or not np.all(pad_values[:, :-1] == 0):
        raise AssertionError("左侧窗口补零错误")
    if not np.all(pad_mask[:, :-1] == 0):
        raise AssertionError("左侧窗口 mask 错误")
    if not np.allclose(pad_values[:, -1], np.asarray(dataset.sequence[0, [0, 1], :])):
        raise AssertionError("窗口末端不是当前时间，可能存在时间泄漏")

    batch = dataset.time_batch(
        "train", TRAIN_START, cap=64, rng=np.random.default_rng(CONFIG.seed), require_target=True
    )
    tensors = to_device(batch, device, include_target=True)
    model.train()
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        prediction = forward_batch(model, tensors)
        loss, loss_parts = composite_loss(prediction, tensors["target"])
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG.gradient_clip)
    if not bool(torch.isfinite(gradient_norm)):
        raise AssertionError("冒烟测试梯度非有限")
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()
    ema.update(model)
    if prediction.shape != (64,) or not bool(torch.isfinite(prediction).all()):
        raise AssertionError("冒烟预测 shape 或有限性错误")

    checkpoint_path = smoke_dir / "checkpoint_smoke.pt"
    history = [{"epoch": 0, **loss_parts}]
    atomic_torch_save(
        checkpoint_path,
        checkpoint_payload(0, 1, model, ema, optimizer, scheduler, scaler, history),
    )
    restored_model = StockMixerLite().to(device)
    restored_ema = ModelEMA(restored_model, CONFIG.ema_decay)
    restored_optimizer = torch.optim.AdamW(
        restored_model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay
    )
    restored_scheduler = torch.optim.lr_scheduler.LambdaLR(
        restored_optimizer, lr_lambda=lambda step: lr_multiplier(step)
    )
    restored_scaler = torch.amp.GradScaler(
        "cuda", enabled=True, init_scale=1024.0, growth_interval=2000
    )
    restored = restore_checkpoint(
        checkpoint_path,
        restored_model,
        restored_ema,
        restored_optimizer,
        restored_scheduler,
        restored_scaler,
        device,
    )
    if restored[:2] != (0, 1):
        raise AssertionError("检查点恢复的 epoch/global_step 错误")

    recursive_checks = run_recursive_self_tests()
    raw_grid, recursive_grid = predict_split(
        dataset, restored_ema.model, "test", device, time_limit=2
    )
    evaluation_mask = dataset.evaluation_mask("test")
    raw_contract = validate_prediction_grid(raw_grid, evaluation_mask)
    recursive_contract = validate_prediction_grid(recursive_grid, evaluation_mask)
    atomic_save_npy(smoke_dir / "prediction_smoke.npy", recursive_grid)
    report = {
        "status": "passed",
        "device": str(device),
        "gpu": torch.cuda.get_device_name(device),
        "torch": torch.__version__,
        "cuda_build": torch.version.cuda,
        "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
        "batch_size": int(batch["stock"].size),
        "loss": loss_parts,
        "gradient_norm": float(gradient_norm.detach().cpu()),
        "checkpoint": str(checkpoint_path),
        "checkpoint_restored": True,
        "recursive_self_tests": recursive_checks,
        "raw_contract": raw_contract,
        "recursive_contract": recursive_contract,
        "note": "仅前两期含模型预测，其余评估位保持 0.5；该文件不可提交。",
    }
    atomic_write_json(smoke_dir / "smoke_report.json", report)
    return report


def build_report(metadata: dict, metrics: dict) -> str:
    valid_raw = metrics["valid_raw"]
    valid_recursive = metrics["valid_recursive"]
    delta = valid_recursive["mean_rankic"] - valid_raw["mean_rankic"]
    return f"""# exp_013 Mask-aware StockMixer-Lite + 递归预测

- 状态：完整训练与推理完成
- 模型：单一 StockMixer-Lite，seed=42，EMA={CONFIG.ema_decay}
- 训练：{CONFIG.epochs} epochs × {CONFIG.steps_per_epoch} 时间截面/epoch
- Valid 原始 RankIC：`{valid_raw['mean_rankic']:.6f}`
- Valid 递归 RankIC：`{valid_recursive['mean_rankic']:.6f}`
- 递归增量：`{delta:+.6f}`
- Test 预测 SHA-256：`{metadata['prediction_sha256']}`
- 当前线上最佳参考：`0.109959`

本实验不融合任何其他模型，也不会自动覆盖 `04_results/final_submission/prediction.npy`。
"""


def run_full(dataset: Dataset, device: torch.device, result_dir: Path, resume: bool) -> dict:
    if device.type != "cuda":
        raise RuntimeError("完整训练固定使用 GPU，但当前 CUDA 不可用")
    result_dir.mkdir(parents=True, exist_ok=True)
    seed_everything(CONFIG.seed)
    model = StockMixerLite().to(device)
    ema = ModelEMA(model, CONFIG.ema_decay)
    started = time.time()
    history = train_full(dataset, model, ema, device, result_dir, resume=resume)
    inference_model = ema.model.to(device).eval()
    valid_raw, valid_recursive = predict_split(dataset, inference_model, "valid", device)
    test_raw, test_recursive = predict_split(dataset, inference_model, "test", device)
    valid_mask = dataset.evaluation_mask("valid")
    test_mask = dataset.evaluation_mask("test")
    contracts = {
        "valid_raw": validate_prediction_grid(valid_raw, valid_mask),
        "valid_recursive": validate_prediction_grid(valid_recursive, valid_mask),
        "test_raw": validate_prediction_grid(test_raw, test_mask),
        "test_recursive": validate_prediction_grid(test_recursive, test_mask),
    }
    if contracts["test_recursive"]["evaluation_count"] != 2_042_538:
        raise AssertionError("Test 评估位置数量不符合官方缓存契约")
    metrics = {
        "valid_raw": score_grid(dataset, "valid", valid_raw),
        "valid_recursive": score_grid(dataset, "valid", valid_recursive),
    }
    metrics["recursive_delta"] = (
        metrics["valid_recursive"]["mean_rankic"] - metrics["valid_raw"]["mean_rankic"]
    )
    atomic_save_npy(result_dir / "valid_prediction_raw.npy", valid_raw)
    atomic_save_npy(result_dir / "valid_prediction.npy", valid_recursive)
    atomic_save_npy(result_dir / "prediction_raw.npy", test_raw)
    atomic_save_npy(result_dir / "prediction.npy", test_recursive)
    prediction_path = result_dir / "prediction.npy"
    metadata = {
        "experiment_id": "exp_013_stockmixer_recursive",
        "status": "completed",
        "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "runtime_seconds": time.time() - started,
        "config": asdict(CONFIG),
        "device": str(device),
        "gpu": torch.cuda.get_device_name(device),
        "torch": torch.__version__,
        "cuda_build": torch.version.cuda,
        "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
        "epochs_completed": len(history),
        "contracts": contracts,
        "prediction_sha256": file_sha256(prediction_path),
        "formal_submission_overwritten": False,
    }
    atomic_write_json(result_dir / "metrics.json", metrics)
    atomic_write_json(result_dir / "metadata.json", metadata)
    atomic_write_text(result_dir / "experiment_report.md", build_report(metadata, metrics))
    return {"metadata": metadata, "metrics": metrics}


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset = Dataset(DATASET_DIR)

if RUN_MODE == "smoke":
    run_result = smoke_test(dataset, device, RESULT_DIR)
else:
    run_result = run_full(dataset, device, RESULT_DIR, resume=RESUME)

print(json.dumps(json_ready(run_result), ensure_ascii=False, indent=2))


epoch 01/20 step 064/512 loss=0.75835 lr=1.90e-05
epoch 01/20 step 128/512 loss=0.71153 lr=3.78e-05
epoch 01/20 step 192/512 loss=0.70921 lr=5.65e-05
epoch 01/20 step 256/512 loss=0.70952 lr=7.53e-05
epoch 01/20 step 320/512 loss=0.67141 lr=9.40e-05
epoch 01/20 step 384/512 loss=0.74868 lr=1.13e-04
epoch 01/20 step 448/512 loss=0.74525 lr=1.32e-04
epoch 01/20 step 512/512 loss=0.71573 lr=1.50e-04
{"epoch": 1, "steps": 512, "samples": 393216, "loss": 0.6933058213908225, "ic_loss": 0.9735125206643716, "huber_loss": 0.020346073961263755, "pairwise_loss": 0.6940784089965746, "learning_rate": 0.00015029296874999998, "seconds": 22.39684295654297}
epoch 02/20 step 064/512 loss=0.60513 lr=1.69e-04
epoch 02/20 step 128/512 loss=0.74263 lr=1.88e-04
epoch 02/20 step 192/512 loss=0.65895 lr=2.07e-04
epoch 02/20 step 256/512 loss=0.67118 lr=2.25e-04
epoch 02/20 step 320/512 loss=0.61782 lr=2.44e-04
epoch 02/20 step 384/512 loss=0.77817 lr=2.63e-04
epoch 02/20 step 448/512 loss=0.62741 lr=2.82e-04
e

## 7. 结果说明

- `smoke` 成功只代表代码、CUDA、反向传播、检查点、递归和输出契约可运行；其预测不可提交。
- `full` 完成后，最终文件位于 `04_results/exp_013_stockmixer_recursive/prediction.npy`。
- 完整训练只报告 Valid 原始/递归 RankIC，不根据 Valid 重新调参。
